In [25]:
!pkill -f flask
!pkill -f ngrok

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [44]:
!ls "/content/drive/MyDrive/Spam_Project"

spam_model.pkl	vectorizer.pkl


In [45]:
!pip install flask flask-ngrok pyngrok

In [78]:
%%writefile app.py

from flask import Flask, render_template, request, redirect, session
import pickle
import sqlite3
import os
import numpy as np

app = Flask(__name__)
app.secret_key = "spam_secret_key"

# ================================
# LOAD MODEL
# ================================
model = pickle.load(open("/content/drive/MyDrive/Spam_Project/spam_model.pkl", "rb"))
vectorizer = pickle.load(open("/content/drive/MyDrive/Spam_Project/vectorizer.pkl", "rb"))

# ================================
# DATABASE
# ================================
def get_db():
    return sqlite3.connect("database.db")

def init_db():
    conn = sqlite3.connect("database.db")
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS users (
            username TEXT PRIMARY KEY,
            password TEXT
        )
    """)
    conn.commit()
    conn.close()

init_db()

# ================================
# HOME
# ================================
@app.route("/")
def home():
    if "user" in session:
        return redirect("/dashboard")
    return redirect("/login")

# ================================
# REGISTER
# ================================
@app.route("/register", methods=["GET", "POST"])
def register():
    if request.method == "POST":
        username = request.form["username"]
        password = request.form["password"]

        conn = get_db()
        cur = conn.cursor()

        cur.execute("SELECT * FROM users WHERE username=?", (username,))
        if cur.fetchone():
            return render_template("register.html", error="User already exists")

        cur.execute("INSERT INTO users VALUES (?,?)", (username, password))
        conn.commit()
        conn.close()

        return redirect("/login")

    return render_template("register.html")

# ================================
# LOGIN
# ================================
@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        username = request.form["username"]
        password = request.form["password"]

        conn = get_db()
        cur = conn.cursor()

        cur.execute("SELECT * FROM users WHERE username=? AND password=?",
                    (username, password))
        data = cur.fetchone()

        if data:
            session["user"] = username
            return redirect("/dashboard")
        else:
            return render_template("login.html", error="Invalid credentials")

    return render_template("login.html")

# ================================
# DASHBOARD
# ================================
@app.route("/dashboard")
def dashboard():
    if "user" not in session:
        return redirect("/login")
    return render_template("dashboard.html", user=session["user"])

# ================================
# PREDICT
# ================================
@app.route("/predict", methods=["POST"])
def predict():
    if "user" not in session:
        return redirect("/login")

    email_text = request.form["email"]

    vector = vectorizer.transform([email_text])
    prediction = model.predict(vector)[0]
    prob = model.predict_proba(vector)[0]

    label = "Spam" if prediction == 0 else "Not Spam"
    confidence = round(np.max(prob) * 100, 2)

    spam_prob = round(prob[0] * 100, 2)
    ham_prob = round(prob[1] * 100, 2)

    return render_template("result.html",
                           text=email_text,
                           prediction=label,
                           confidence=confidence,
                           spam_prob=spam_prob,
                           ham_prob=ham_prob)

# ================================
# LOGOUT
# ================================
@app.route("/logout")
def logout():
    session.clear()
    return redirect("/login")

# ================================
# RUN
# ================================
if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8000)

Overwriting app.py


In [79]:
!mkdir -p templates
!mkdir -p static

In [80]:
%%writefile templates/login.html

<!DOCTYPE html>
<html>
<head>
<title>Login</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>

<div class="card">
<h2>Login</h2>

{% if error %}
<p class="error">{{ error }}</p>
{% endif %}

<form method="POST">
<input type="text" name="username" placeholder="Username" required>
<input type="password" name="password" placeholder="Password" required>
<button>Login</button>
</form>

<a href="/register">Create Account</a>

</div>

</body>
</html>

Overwriting templates/login.html


In [81]:
%%writefile templates/register.html
<!DOCTYPE html>
<html>
<head>
<title>Register</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>

<div class="card">
<h2>Register</h2>

{% if error %}
<p class="error">{{ error }}</p>
{% endif %}

<form method="POST">
<input type="text" name="username" required placeholder="Username">
<input type="password" name="password" required placeholder="Password">
<button>Create Account</button>
</form>

</div>

</body>
</html>

Overwriting templates/register.html


In [82]:
%%writefile templates/dashboard.html
<!DOCTYPE html>
<html>
<head>
<title>Spam Detector</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>

<nav>
<h2>📧 Spam Detector AI</h2>
<a href="/logout">Logout</a>
</nav>

<div class="container">
<h2>Welcome {{ user }}</h2>

<form action="/predict" method="POST">
<textarea name="email" placeholder="Paste email here..." required></textarea>
<button>Analyze Email</button>
</form>

</div>

</body>
</html>

Overwriting templates/dashboard.html


In [83]:
%%writefile templates/result.html
<!DOCTYPE html>
<html>
<head>
<title>Result</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>

<div class="container">

<h2>Result</h2>

<p>{{ text }}</p>

<h1>{{ prediction }}</h1>
<p>Confidence: {{ confidence }}%</p>

<div class="bar">
<div class="spam" style="width: {{ spam_prob }}%"></div>
</div>
<p>Spam: {{ spam_prob }}%</p>
<p>Not Spam: {{ ham_prob }}%</p>

<a href="/dashboard">Try Again</a>

</div>

</body>
</html>""

Overwriting templates/result.html


In [84]:
%%writefile static/style.css
body {
    font-family: Arial;
    background: #0f172a;
    color: white;
}

.card, .container {
    max-width: 500px;
    margin: auto;
    margin-top: 80px;
    padding: 30px;
    background: #1e293b;
    border-radius: 10px;
}

input, textarea {
    width: 100%;
    padding: 10px;
    margin-top: 10px;
}

button {
    margin-top: 20px;
    width: 100%;
    padding: 10px;
    background: #3b82f6;
    border: none;
    color: white;
}

.bar {
    height: 10px;
    background: #444;
    margin-top: 10px;
}

.spam {
    height: 10px;
    background: red;
}

.error {
    color: red;
}

Overwriting static/style.css


In [85]:
# ============================
# ✅ Run Flask in background
# ============================
!pkill -f flask


In [86]:
!lsof -i :8000

In [87]:

!kill -9 1992

/bin/bash: line 1: kill: (1992) - No such process


In [88]:
!nohup python app.py > log.txt 2>&1 &

In [57]:
!pip install pyngrok

In [89]:
!tail -n 20 log.txt

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 1.2.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.2.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.2.2 when using version 1.6.1. This might lead to breaking code o

In [90]:
from pyngrok import ngrok

ngrok.set_auth_token("3ApYiAvCLUh52bnAvVdimdQ2QfU_2tBZUG86Y2UuaVJm3DVvX")

public_url = ngrok.connect(8000)
print(public_url)

NgrokTunnel: "https://reina-pretraditional-jacqualine.ngrok-free.dev" -> "http://localhost:8000"
